# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
#
from google.colab import userdata
from datasets import load_dataset

hf_token = userdata.get('HF_token')
dataset_content = load_dataset("FlyRank/internship-warehouse", 'dim_content', token=hf_token)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

In [ ]:
#Stream Large Datasets: If the dataset is too big for Colab's RAM, add streaming=True to download data on-the-fly instead of all at once:python
streamed_data = load_dataset("FlyRank/internship-warehouse", split="train", streaming=True)
# View the first item
print(next(iter(streamed_data)))

In [14]:
#iterate it in batches
import pandas as pd

# Assuming 'dataset' is a DatasetDict and contains a 'train' split
# If your dataset has a different split name, please adjust 'train' accordingly.
if isinstance(dataset, dict) and 'train' in dataset:
    data_split = dataset['train']
elif hasattr(dataset, 'keys') and len(dataset.keys()) == 1:
    # If it's a DatasetDict with only one split, use that split
    data_split = dataset[list(dataset.keys())[0]]
else:
    # Otherwise, assume 'dataset' itself is the Dataset object
    data_split = dataset

batch_size = 1000  # Define your desired batch size

print(f"Iterating over {len(data_split)} records in batches of {batch_size}...")

# Iterate over the dataset in batches and convert to pandas DataFrames
for i in range(0, len(data_split), batch_size):
    batch_data = data_split[i:i + batch_size]
    df_batch = pd.DataFrame(batch_data)
    print(f"--- Processing batch {i // batch_size + 1} ---")
    display(df_batch.head()) # Display the head of the current batch DataFrame
    # You can now perform operations on df_batch
    # For demonstration, we'll only show the first batch.
    break


Iterating over 78835655 records in batches of 1000...
--- Processing batch 1 ---


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [6]:
#for getting selected columns from large dataset
import duckdb

con = duckdb.connect()

# Replace with your HF read token
con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{hf_token}'
)
""")

In [9]:
df = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
display(df.head())

,client_hash_id,content_hash_id
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1,client_73cda7b4e4f265ea,content_05597932fe4da067
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916
3,client_73cda7b4e4f265ea,content_905aa32a0230694e
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

 unit of analysis is a 'content_hash_id' on a specific date from Jan 1 – Mar 31 for training. For evaluating next 3 month data can be used

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

# Task
The goal is to define a data contract for analyzing ranking visibility based on the `dim_content` dataset. This involves categorizing all available fields into label, contextual/identifier, potential features, and excluded fields, along with clear justifications for each classification. The primary focus is on identifying variables that influence or are relevant to `gsc_avg_position`.

## List All Available Fields

### Subtask:
List all available fields from the `dim_content` dataset.


**Reasoning**:
To list all available fields, I will access the 'train' split of the `dataset_content` object and print its column names.



In [15]:
print(dataset_content['train'].column_names)

NameError: name 'dataset_content' is not defined

**Reasoning**:
To resolve the `NameError`, I need to ensure the `dataset_content` variable is defined by re-executing the dataset loading code. Then, I will print the column names from the 'train' split.



In [16]:
from google.colab import userdata
from datasets import load_dataset

hf_token = userdata.get('HF_token')
dataset_content = load_dataset("FlyRank/internship-warehouse", 'dim_content', token=hf_token)

print(dataset_content['train'].column_names)

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


## Identify the Label (Target) Variable

### Subtask:
Explicitly confirm `gsc_avg_position` as the target variable for ranking visibility analysis.


```markdown
## Label (Target) Variable

The label, or target variable, for this analysis is `gsc_avg_position`. This metric is crucial for understanding ranking visibility, as it directly indicates the average position of content in Google Search results. A lower average position signifies better visibility.

It is important to note that `gsc_avg_position` was not present in the fields listed for the `dim_content` dataset in the previous step. For a comprehensive analysis, this variable would need to be obtained from a different dataset, such as `fact_content_daily_performance`, and joined with the `dim_content` data as needed.
```

## Identify Contextual/Identifier Fields

### Subtask:
Identify which fields serve as unique identifiers or provide essential context but are not directly used as predictive features.


```markdown
### Identified Contextual/Identifier Fields:

Based on the `dim_content` dataset fields, the following are identified as contextual or identifier fields:

*   `client_hash_id`: Unique identifier for each client. While essential for filtering or joining, it does not directly represent a predictive feature for content ranking.
*   `content_hash_id`: Unique identifier for each piece of content. This forms the primary unit of analysis and is crucial for linking data but is not a predictive feature itself.
*   `keyword_hash_id`: Unique identifier for each keyword. Similar to `content_hash_id`, it's used for linking and identifying specific keywords.
*   `url_hash_id`: Unique identifier for each URL associated with the content. Useful for tracking but not a direct predictive feature.
*   `content_created_date`: The date when the content was created. Provides historical context but is not typically used as a direct predictive feature; rather, age of content derived from it might be.
*   `content_updated_date`: The date when the content was last updated. Similar to `content_created_date`, it provides temporal context.
*   `keyword_created_date`: The date when the keyword was created. Contextual information.
*   `last_optimized_date`: The date the content was last optimized. Provides a timestamp for optimization efforts, offering contextual information about content lifecycle.
*   `optimization_eligible_date`: The date the content became eligible for optimization. Provides contextual information related to content management.

These fields are vital for data integrity, joining datasets, and filtering, but they are not directly used as independent variables to predict `gsc_avg_position`.
```

## Identify Potential Feature Fields

### Subtask:
Go through the remaining fields from the `dim_content` dataset and select those that could influence or correlate with `gsc_avg_position`. These will serve as independent variables.


```markdown
### Identified Potential Feature Fields:

Based on the remaining fields from the `dim_content` dataset, after excluding identifier and contextual fields, the following are considered potential feature fields that could influence or correlate with `gsc_avg_position`:

*   `keyword_char_count`: The length of the keyword in characters. This could influence ranking as shorter/longer keywords might indicate different search intent or competitiveness.
*   `keyword_token_count`: The number of words (tokens) in the keyword. Similar to character count, this can reflect the specificity and long-tail nature of a keyword, which impacts search performance.
*   `url_char_count`: The length of the URL in characters. URL length and structure can sometimes be a minor ranking factor.
*   `content_type`: Categorical field indicating the type of content (e.g., blog post, product page). Different content types may perform differently in search results.
*   `search_volume`: The average monthly searches for the keyword. Higher search volume keywords are generally more competitive and harder to rank for, but also offer higher potential traffic.
*   `competition`: A metric indicating the level of competition for the keyword. Higher competition generally means it's more challenging to rank well.
*   `competition_level`: Categorical level of competition (e.g., 'Low', 'Medium', 'High'). Provides a qualitative measure of keyword difficulty.
*   `cpc`: Cost Per Click for the keyword. Can be an indicator of commercial intent and keyword value, which might correlate with how aggressively businesses optimize for it, indirectly affecting ranking.
*   `main_intent`: The primary search intent behind the keyword (e.g., informational, navigational, commercial). Matching content intent with search intent is crucial for ranking.
*   `backlinks`: The number of backlinks pointing to the content. Backlinks are a strong signal of authority and are a critical factor in search engine ranking.
*   `category_count`: The number of categories associated with the content. This could reflect content organization and topical breadth, potentially influencing its relevance.
*   `provider_used`: The content provider or source. Different providers might have varying quality standards or authority.
*   `model_used`: The model used for content generation or optimization. Could indicate the sophistication or automation level in content creation, which might impact quality.
*   `char_count`: The total character count of the content. Content length can sometimes correlate with comprehensiveness, which is a ranking factor.
*   `word_count`: The total word count of the content. Similar to character count, longer content *can* be perceived as more comprehensive and authoritative.
*   `is_published`: A boolean indicating whether the content is published. Only published content can rank.
*   `is_deleted`: A boolean indicating whether the content has been deleted. Deleted content will not rank.

These features provide a comprehensive set of variables to model the factors influencing a content piece's average search position.
```

## Identify Excluded Fields and Rationale

### Subtask:
List any fields that should be excluded from your analysis and provide a clear reason for their exclusion.


```markdown
### Identified Excluded Fields and Rationale:

Based on the comprehensive listing of fields from the `dim_content` dataset and their subsequent categorization into 'Label (Target) Variable' (conceptually `gsc_avg_position` from another dataset), 'Contextual/Identifier Fields', and 'Potential Feature Fields', all fields present in `dim_content` have been accounted for.

Therefore, within the scope of the `dim_content` dataset, there are **no fields that need to be explicitly excluded** from the analysis based on direct criteria such as data leakage or irrelevance, as all have been deemed either identifiers/context or potential features.

However, it's crucial to acknowledge the principle of excluding fields that could lead to data leakage or are redundant. For instance, if `gsc_sum_position` were present in the `dim_content` dataset and `gsc_avg_position` was the target variable, then `gsc_sum_position` would be a prime candidate for exclusion. This is because `gsc_avg_position` is often derived directly from `gsc_sum_position` (e.g., `gsc_avg_position = gsc_sum_position / gsc_impressions`). Including such a highly correlated, or even deterministically linked, variable as a feature would cause data leakage, artificially inflating model performance and leading to an unrealistic assessment of predictive power. Since `gsc_avg_position` itself is not in `dim_content`, this specific example serves as a conceptual exclusion criterion for future data integration.
```

## Final Task

### Subtask:
Summarize the categorized fields, including the label, context/identifiers, selected features, and excluded fields with their justifications, to establish a clear data contract for your analysis.
